In [ ]:
# 安裝必要套件（Colab 環境）
!pip install transformers torch scikit-learn pandas matplotlib seaborn -q
print("套件安裝完成！")

套件安裝完成！


In [ ]:
# 從 GitHub 下載公開資料集（不需要 Google Drive）
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/veripromiseesg/veripromiseesgdataset/ac91c1c8b5d116edf6fc44cccc1ee3b618f5a207/vpesg4ktrain1000v1.json"
urllib.request.urlretrieve(DATA_URL, "vpesg4k_train_1000.json")
print("資料下載完成！")

資料下載完成！


In [ ]:
# 複製一份避免改到原始資料
data = df.copy()

# 文字欄位補空字串
text_cols = [
    "data", "esg_type", "promise_status", "promise_string",
    "verification_timeline", "evidence_status", "evidence_string",
    "evidence_quality"
]

for col in text_cols:
    if col in data.columns:
        data[col] = data[col].fillna("").astype(str).str.strip()

print(data[text_cols].head())

                                                data esg_type promise_status  \
0  聯發科技除在「工作規則」中依照勞基法明確規定「員工在產假期間公司不得終止勞動契約」外，為支持...        S            Yes   
1  在與供應商攜手邁向永續發展的過程中，台泥致力於建立一套以合作為基礎的價值體體系。核心原則是推...        E            Yes   
2  III. 產品危害物質減免 (HSF) 短、中、長期目標設定    **1. 2024 年 ...        E            Yes   
3  面對這些全球與國內政策的推動，富邦人壽秉持永續經營理念，積極透過永續金融的投資力量，引導被投...        E            Yes   
4  關注人才關鍵議題，新加坡亞洲新聞臺（CNA）特別製作專題《Taiwan’s tech ind...        S            Yes   

                                      promise_string  verification_timeline  \
0  為支持同仁與其家人度過人生不同階段，自 2024 年起提供女性員工在分娩前後計有 12 週共...                already   
1                             台泥致力於建立一套以合作為基礎的價值體體系。  between_2_and_5_years   
2                     III. 產品危害物質減免 (HSF) 短、中、長期目標設定  between_2_and_5_years   
3           秉持永續經營理念，積極透過永續金融的投資力量，引導被投資公司與企業進行永續轉型，                already   
4  透過教育和相關計畫，致力於提升女性在 STEM 領域的參與，確保科技創新能夠從更多元的觀點中...                already   

  evidence_status                           

In [ ]:
def normalize_esg_type(esg):
    if not esg or esg.strip() == "":
        return "Unknown"

    parts = [x.strip() for x in esg.split(";") if x.strip() != ""]

    # 去重複後排序，確保 E;S 和 S;E 變成同一種
    parts = sorted(set(parts))

    return ";".join(parts)

data["esg_type_norm"] = data["esg_type"].apply(normalize_esg_type)

print(data[["esg_type", "esg_type_norm"]].drop_duplicates().sort_values("esg_type_norm").head(20))

    esg_type esg_type_norm
1          E             E
45       E;G           E;G
49     E;S;G         E;G;S
442    G;S;E         E;G;S
642    G;E;S         E;G;S
724    S;E;G         E;G;S
33       E;S           E;S
483      S;E           E;S
12         G             G
11       G;S           G;S
161      S;G           G;S
0          S             S
21                 Unknown


In [ ]:
print("=== ESG 類別分布 ===")
print(data["esg_type_norm"].value_counts(dropna=False))

print("\n=== promise_status 分布 ===")
print(data["promise_status"].value_counts(dropna=False))

print("\n=== evidence_quality 分布 ===")
print(data["evidence_quality"].value_counts(dropna=False))

print("\n=== verification_timeline 分布 ===")
print(data["verification_timeline"].value_counts(dropna=False))

=== ESG 類別分布 ===
esg_type_norm
S          373
E          326
G          241
E;G;S       21
G;S         16
E;S         12
Unknown      9
E;G          2
Name: count, dtype: int64

=== promise_status 分布 ===
promise_status
Yes    814
No     186
Name: count, dtype: int64

=== evidence_quality 分布 ===
evidence_quality
Clear         552
N/A           323
Not Clear     124
Misleading      1
Name: count, dtype: int64

=== verification_timeline 分布 ===
verification_timeline
already                  366
between_2_and_5_years    238
longer_than_5_years      197
N/A                      186
within_2_years            13
Name: count, dtype: int64


In [ ]:
q1 = pd.crosstab(
    data["esg_type_norm"],
    data["promise_status"],
    normalize="index"
) * 100

q1 = q1.round(2)

print("=== [Q1] 各類型與承諾比例（promise_status）分析 ===")
display(q1)

=== [Q1] 各類型與承諾比例（promise_status）分析 ===


promise_status,No,Yes
esg_type_norm,,
E,17.48,82.52
E;G,0.00,100.00
E;G;S,19.05,80.95
E;S,16.67,83.33
G,24.48,75.52
G;S,0.00,100.00
S,16.09,83.91
Unknown,44.44,55.56


In [ ]:
q2 = pd.crosstab(
    data["esg_type_norm"],
    data["evidence_quality"],
    normalize="index"
) * 100

q2 = q2.round(2)

print("=== [Q2] 各類型與佐證品質（evidence_quality）分析 ===")
display(q2)

=== [Q2] 各類型與佐證品質（evidence_quality）分析 ===


evidence_quality,Clear,Misleading,N/A,Not Clear
esg_type_norm,,,,
E,56.75,0.00,31.60,11.66
E;G,100.00,0.00,0.00,0.00
E;G;S,42.86,0.00,47.62,9.52
E;S,33.33,0.00,41.67,25.00
G,46.47,0.41,37.76,15.35
G;S,81.25,0.00,18.75,0.00
S,59.79,0.00,28.42,11.80
Unknown,44.44,0.00,55.56,0.00


In [ ]:
q3 = pd.crosstab(
    data["esg_type_norm"],
    data["verification_timeline"],
    normalize="index"
) * 100

q3 = q3.round(2)

print("=== [Q3] 各類型與承諾時程（verification_timeline）分析 ===")
display(q3)

=== [Q3] 各類型與承諾時程（verification_timeline）分析 ===


verification_timeline,N/A,already,between_2_and_5_years,longer_than_5_years,within_2_years
esg_type_norm,,,,,
E,17.48,24.85,25.77,29.75,2.15
E;G,0.00,0.00,100.00,0.00,0.00
E;G;S,19.05,23.81,38.10,19.05,0.00
E;S,16.67,41.67,25.00,16.67,0.00
G,24.48,39.42,19.09,16.18,0.83
G;S,0.00,31.25,18.75,43.75,6.25
S,16.09,46.38,24.40,12.33,0.80
Unknown,44.44,22.22,11.11,22.22,0.00


In [ ]:
def esg_group(esg):
    if esg == "Unknown":
        return "Unknown"
    elif ";" in esg:
        return "Multi"
    else:
        return "Single"

data["esg_group"] = data["esg_type_norm"].apply(esg_group)

single_multi_promise = pd.crosstab(
    data["esg_group"],
    data["promise_status"],
    normalize="index"
) * 100

print("=== 單一 / 複合 ESG 類別 vs promise_status ===")
display(single_multi_promise.round(2))

=== 單一 / 複合 ESG 類別 vs promise_status ===


promise_status,No,Yes
esg_group,,
Multi,11.76,88.24
Single,18.72,81.28
Unknown,44.44,55.56


In [ ]:
evidence_status_analysis = pd.crosstab(
    data["esg_type_norm"],
    data["evidence_status"],
    normalize="index"
) * 100

print("=== 各類型 vs evidence_status ===")
display(evidence_status_analysis.round(2))

=== 各類型 vs evidence_status ===


evidence_status,N/A,No,Yes
esg_type_norm,,,
E,17.48,14.11,68.40
E;G,0.00,0.00,100.00
E;G;S,19.05,28.57,52.38
E;S,16.67,25.00,58.33
G,24.48,13.28,62.24
G;S,0.00,18.75,81.25
S,16.09,12.33,71.58
Unknown,44.44,11.11,44.44


In [ ]:
not_clear_rank = q2[["Not Clear"]].sort_values("Not Clear", ascending=False) if "Not Clear" in q2.columns else None

print("=== Not Clear 排名 ===")
display(not_clear_rank)

=== Not Clear 排名 ===


evidence_quality,Not Clear
esg_type_norm,
E;S,25.00
G,15.35
S,11.80
E,11.66
E;G;S,9.52
E;G,0.00
G;S,0.00
Unknown,0.00


In [ ]:
rules = []

if "Yes" in q1.columns:
    for esg_type, row in q1.iterrows():
        yes_ratio = row.get("Yes", 0)
        no_ratio = row.get("No", 0)

        if yes_ratio >= 85:
            rules.append(f"{esg_type} 類文本承諾比例高（Yes={yes_ratio:.2f}%），可視為高承諾傾向類型。")
        elif no_ratio >= 30:
            rules.append(f"{esg_type} 類文本不承諾比例偏高（No={no_ratio:.2f}%），需提高承諾判定門檻。")

In [ ]:
if "Not Clear" in q2.columns:
    for esg_type, row in q2.iterrows():
        not_clear_ratio = row.get("Not Clear", 0)
        clear_ratio = row.get("Clear", 0)

        if not_clear_ratio >= 30:
            rules.append(f"{esg_type} 類文本佐證較模糊（Not Clear={not_clear_ratio:.2f}%），需加強數據與時間資訊檢查。")
        if clear_ratio >= 60:
            rules.append(f"{esg_type} 類文本佐證較明確（Clear={clear_ratio:.2f}%），適合使用量化特徵判定。")

In [ ]:
for esg_type, row in q3.iterrows():
    already_ratio = row.get("already", 0)
    mid_ratio = row.get("between_2_and_5_years", 0)
    long_ratio = row.get("longer_than_5_years", 0)

    if already_ratio >= 45:
        rules.append(f"{esg_type} 類文本偏向描述已執行事項（already={already_ratio:.2f}%）。")
    if long_ratio >= 30:
        rules.append(f"{esg_type} 類文本偏向長期承諾（longer_than_5_years={long_ratio:.2f}%）。")
    if mid_ratio >= 35:
        rules.append(f"{esg_type} 類文本偏向中期規劃（between_2_and_5_years={mid_ratio:.2f}%）。")

In [ ]:
print("=== 自動萃取的規則候選 ===")
for i, r in enumerate(rules, 1):
    print(f"{i}. {r}")

=== 自動萃取的規則候選 ===
1. E;G 類文本承諾比例高（Yes=100.00%），可視為高承諾傾向類型。
2. G;S 類文本承諾比例高（Yes=100.00%），可視為高承諾傾向類型。
3. Unknown 類文本不承諾比例偏高（No=44.44%），需提高承諾判定門檻。
4. E;G 類文本佐證較明確（Clear=100.00%），適合使用量化特徵判定。
5. G;S 類文本佐證較明確（Clear=81.25%），適合使用量化特徵判定。
6. E;G 類文本偏向中期規劃（between_2_and_5_years=100.00%）。
7. E;G;S 類文本偏向中期規劃（between_2_and_5_years=38.10%）。
8. G;S 類文本偏向長期承諾（longer_than_5_years=43.75%）。
9. S 類文本偏向描述已執行事項（already=46.38%）。


In [ ]:
!pip install jieba -q

In [ ]:
import re
import math
import jieba
import pandas as pd
from collections import Counter, defaultdict

# =========================
# 0. 基本設定
# =========================

# 你要分析的欄位
TEXT_COL = "data"
LABEL_COL = "esg_type_norm" # 將 'esg_type_raw' 修改為 'esg_type_norm'

# 如果你只想看這幾類，可以改這裡
TARGET_LABELS = ["Unknown", "E", "G", "S"]

# 可自行擴充停用詞
stopwords = set([
    "公司", "企業", "集團", "相關", "進行", "推動", "提升", "提供", "發展", "管理",
    "永續", "使用", "措施", "透過", "工作", "服務", "合作", "環境", "風險", "安全",
    "2024", "本公司", "本集團", "今年", "年度", "以及", "因此", "其中", "包括",
    "針對", "持續", "積極", "部分", "方式", "內容", "目標", "執行", "評估",
    "重大", "資訊", "報告", "全球", "企業", "公司", "進行", "相關", "發展",
    "使用", "管理", "永續", "措施", "代表", "地區", "市場"
])

# 可視需要保留數字型年份，若不想保留就設 False
KEEP_YEAR = False

# 類別數門檻：若某詞出現在太多類別，就視為通用詞
# 例如四類中出現在 >=3 類，就排除
COMMON_WORD_CATEGORY_THRESHOLD = 3

# 每個類別輸出前幾個關鍵字
TOP_K = 30


# =========================
# 1. 前處理函式
# =========================

def clean_text(text: str) -> str:
    text = str(text).strip()
    # 保留中文、英文、數字，其他符號改空白
    text = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def is_valid_token(token: str) -> bool:
    token = token.strip()
    if token == "":
        return False

    # 去掉停用詞
    if token in stopwords:
        return False

    # 去掉長度過短
    if len(token) <= 1:
        return False

    # 去掉純英文
    if re.fullmatch(r"[A-Za-z]+", token):
        return False

    # 去掉純數字
    if re.fullmatch(r"\d+", token):
        if KEEP_YEAR and token in {"2024", "2025", "2030", "2050"}:
            return True
        return False

    return True

def tokenize(text: str):
    text = clean_text(text)
    tokens = jieba.lcut(text)
    tokens = [t.strip() for t in tokens if is_valid_token(t)]
    return tokens


# =========================
# 2. 只保留目標類別
# =========================

df_kw = data.copy()
df_kw = df_kw[df_kw[LABEL_COL].isin(TARGET_LABELS)].copy()

print("分析資料筆數：", df_kw.shape[0])
print("類別分布：")
print(df_kw[LABEL_COL].value_counts())


# =========================
# 3. 各類別斷詞
# =========================

label_to_tokens = defaultdict(list)
label_to_doc_count = Counter()

for _, row in df_kw.iterrows():
    label = row[LABEL_COL]
    text = row[TEXT_COL]
    tokens = tokenize(text)

    label_to_tokens[label].extend(tokens)
    label_to_doc_count[label] += 1


# =========================
# 4. 各類別詞頻
# =========================

label_word_counter = {}
for label in TARGET_LABELS:
    label_word_counter[label] = Counter(label_to_tokens[label])


# =========================
# 5. 找出「通用詞」
#    一個詞若出現在很多類別，就不夠有區分力
# =========================

word_in_labels = defaultdict(set)

for label in TARGET_LABELS:
    for word in label_word_counter[label].keys():
        word_in_labels[word].add(label)

common_words = set()
for word, labels in word_in_labels.items():
    if len(labels) >= COMMON_WORD_CATEGORY_THRESHOLD:
        common_words.add(word)

print("\n被判定為通用詞（前50個）:")
print(sorted(list(common_words))[:50])


# =========================
# 6. 萃取每類的「專屬關鍵字」
#    用簡單版本：
#    score = 本類詞頻 / (其他類平均詞頻 + 1)
# =========================

all_labels = TARGET_LABELS

label_keyword_scores = {}

for target_label in all_labels:
    scores = []
    target_counter = label_word_counter[target_label]

    for word, target_freq in target_counter.items():
        # 排除通用詞
        if word in common_words:
            continue

        other_freqs = []
        for other_label in all_labels:
            if other_label == target_label:
                continue
            other_freqs.append(label_word_counter[other_label].get(word, 0))

        other_avg = sum(other_freqs) / len(other_freqs) if len(other_freqs) > 0 else 0

        # 區分力分數
        score = target_freq / (other_avg + 1)

        # 你也可以加上最低頻率門檻，避免只出現 1 次的字被撈上來
        if target_freq >= 3:
            scores.append((word, target_freq, round(other_avg, 2), round(score, 2)))

    # 依照 score 排序，再依 target_freq 排序
    scores = sorted(scores, key=lambda x: (x[3], x[1]), reverse=True)
    label_keyword_scores[target_label] = scores


# =========================
# 7. 顯示每個類別的代表詞
# =========================

for label in all_labels:
    print(f"\n===== {label} 的代表性關鍵字 =====")
    print("(詞, 本類頻率, 其他類平均頻率, 區分分數)")
    for item in label_keyword_scores[label][:TOP_K]:
        print(item)

分析資料筆數： 949
類別分布：
esg_type_norm
S          373
E          326
G          241
Unknown      9
Name: count, dtype: int64

被判定為通用詞（前50個）:
['一位', '一個', '一同', '一年', '一次', '一步', '一致', '一致性', '一般', '一起', '一項', '三個', '三大', '三年', '三項', '上升', '上市', '上海', '上游', '下列', '不僅', '不利', '不同', '不斷', '不會', '不當', '世代', '世界', '世芯', '並且', '並以', '並依', '並依據', '並協助', '並參考', '並在', '並將', '並對', '並已', '並承諾', '並持續', '並擔', '並由', '並發展', '並確', '並結合', '並與', '並落實', '並規劃', '並訂定']

===== Unknown 的代表性關鍵字 =====
(詞, 本類頻率, 其他類平均頻率, 區分分數)
('勞工', 4, 9.33, 0.39)

===== E 的代表性關鍵字 =====
(詞, 本類頻率, 其他類平均頻率, 區分分數)
('廢棄物', 63, 0.33, 47.25)
('綠建築', 27, 0.0, 27.0)
('用水', 26, 0.0, 26.0)
('建築', 20, 0.0, 20.0)
('零排放', 32, 0.67, 19.2)
('廢水', 19, 0.0, 19.0)
('綠電', 18, 0.0, 18.0)
('電量', 17, 0.0, 17.0)
('廢棄', 17, 0.0, 17.0)
('海洋', 25, 0.67, 15.0)
('燃料', 20, 0.33, 15.0)
('RE100', 19, 0.33, 14.25)
('海岸', 14, 0.0, 14.0)
('年前', 14, 0.0, 14.0)
('自用', 13, 0.0, 13.0)
('環境部', 17, 0.33, 12.75)
('碳足跡', 20, 0.67, 12.0)
('減排', 16, 0.33, 12.0)
('垃圾', 12, 0.0,

In [ ]:
!pip install jieba -q

import re
import jieba
import pandas as pd
from collections import Counter, defaultdict

# =========================
# 0. 基本設定
# =========================
TEXT_COL = "data"
LABEL_COL = "promise_status"
TARGET_LABELS = ["Yes", "No"]

stopwords = set([
    "公司", "企業", "集團", "相關", "進行", "推動", "提升", "提供", "發展", "管理",
    "永續", "使用", "措施", "透過", "工作", "服務", "合作", "環境", "風險", "安全",
    "2024", "本公司", "本集團", "今年", "年度", "以及", "因此", "其中", "包括",
    "針對", "持續", "積極", "部分", "方式", "內容", "執行", "評估", "企業",
    "公司", "相關", "進行", "發展", "使用", "管理", "報告", "全球"
])

KEEP_YEAR = False
COMMON_WORD_CATEGORY_THRESHOLD = 2   # Yes / No 兩類中若兩邊都有，就算通用詞
TOP_K = 30

# =========================
# 1. 前處理函式
# =========================
def clean_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def is_valid_token(token: str) -> bool:
    token = token.strip()
    if token == "":
        return False
    if token in stopwords:
        return False
    if len(token) <= 1:
        return False
    if re.fullmatch(r"[A-Za-z]+", token):
        return False
    if re.fullmatch(r"\d+", token):
        if KEEP_YEAR and token in {"2024", "2025", "2030", "2050"}:
            return True
        return False
    return True

def tokenize(text: str):
    text = clean_text(text)
    tokens = jieba.lcut(text)
    return [t for t in tokens if is_valid_token(t)]

# =========================
# 2. 只保留 Yes / No
# =========================
df_kw = data.copy()
df_kw = df_kw[df_kw[LABEL_COL].isin(TARGET_LABELS)].copy()

print("分析資料筆數：", df_kw.shape[0])
print(df_kw[LABEL_COL].value_counts())

# =========================
# 3. 各類別斷詞
# =========================
label_to_tokens = defaultdict(list)

for _, row in df_kw.iterrows():
    label = row[LABEL_COL]
    text = row[TEXT_COL]
    tokens = tokenize(text)
    label_to_tokens[label].extend(tokens)

# =========================
# 4. 各類別詞頻
# =========================
label_word_counter = {}
for label in TARGET_LABELS:
    label_word_counter[label] = Counter(label_to_tokens[label])

print("\n===== Yes top words =====")
print(label_word_counter["Yes"].most_common(50))

print("\n===== No top words =====")
print(label_word_counter["No"].most_common(50))

# =========================
# 5. 找共同詞（兩邊都常出現）
# =========================
word_in_labels = defaultdict(set)

for label in TARGET_LABELS:
    for word in label_word_counter[label].keys():
        word_in_labels[word].add(label)

common_words = set()
for word, labels in word_in_labels.items():
    if len(labels) >= COMMON_WORD_CATEGORY_THRESHOLD:
        common_words.add(word)

print("\n===== 共同詞（Yes/No 都有）前50 =====")
print(sorted(list(common_words))[:50])

# =========================
# 6. 萃取 Yes / No 專屬關鍵字
# score = 本類頻率 / (另一類頻率 + 1)
# =========================
keyword_scores = {}

for target_label in TARGET_LABELS:
    other_label = "No" if target_label == "Yes" else "Yes"
    scores = []

    for word, target_freq in label_word_counter[target_label].items():
        if word in common_words:
            continue

        other_freq = label_word_counter[other_label].get(word, 0)
        score = target_freq / (other_freq + 1)

        if target_freq >= 5:
            scores.append((word, target_freq, other_freq, round(score, 2)))

    scores = sorted(scores, key=lambda x: (x[3], x[1]), reverse=True)
    keyword_scores[target_label] = scores

print("\n===== Yes 專屬關鍵字 =====")
print("(詞, Yes頻率, No頻率, 區分分數)")
for item in keyword_scores["Yes"][:TOP_K]:
    print(item)

print("\n===== No 專屬關鍵字 =====")
print("(詞, No頻率, Yes頻率, 區分分數)")
for item in keyword_scores["No"][:TOP_K]:
    print(item)

分析資料筆數： 1000
promise_status
Yes    814
No     186
Name: count, dtype: int64

===== Yes top words =====
[('目標', 368), ('董事', 257), ('客戶', 188), ('我們', 184), ('資訊', 177), ('策略', 174), ('氣候', 174), ('金融', 172), ('經營', 156), ('政策', 144), ('社會', 141), ('議題', 136), ('減碳', 132), ('重大', 131), ('產品', 130), ('同仁', 127), ('單位', 126), ('計畫', 125), ('能源', 124), ('未來', 123), ('技術', 122), ('活動', 121), ('資源', 120), ('建立', 119), ('教育', 118), ('內部', 118), ('科技', 116), ('治理', 116), ('每年', 116), ('營運', 115), ('機制', 114), ('同時', 113), ('降低', 112), ('定期', 110), ('委員會', 109), ('轉型', 108), ('綠色', 106), ('行動', 106), ('健康', 106), ('改善', 102), ('員工', 101), ('作業', 100), ('資料', 100), ('致力', 99), ('影響', 99), ('國際', 99), ('氣體', 97), ('訓練', 97), ('主管', 96), ('各項', 96)]

===== No top words =====
[('董事', 52), ('重大', 49), ('議題', 45), ('排放', 40), ('主管', 39), ('包含', 35), ('影響', 34), ('單位', 33), ('處理', 33), ('揭露', 32), ('減少', 32), ('客戶', 31), ('能源', 29), ('金融', 29), ('經濟', 28), ('衝擊', 28), ('資訊', 27), ('健康', 27), ('社會', 26

In [ ]:
import re
import jieba
import pandas as pd
from collections import Counter, defaultdict

# =========================
# 0. 設定
# =========================
TEXT_COL = "data"
LABEL_COL = "evidence_quality"

TARGET_LABELS = ["Clear", "Not Clear", "Misleading", "N/A"]

TOP_K = 30

stopwords = set([
    "公司","企業","集團","相關","進行","推動","提升","提供","發展","管理",
    "永續","使用","措施","透過","工作","服務","合作","環境","風險","安全",
    "2024","本公司","本集團","今年","年度","以及","因此","其中","包括",
    "針對","持續","積極","部分","方式","內容","執行","評估","相關",
    "進行","發展","使用","管理","報告","全球","我們"
])

# =========================
# 1. 前處理
# =========================
def clean_text(text):
    text = str(text)
    text = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    text = clean_text(text)
    tokens = jieba.lcut(text)
    tokens = [t for t in tokens if len(t) > 1 and t not in stopwords and not t.isdigit()]
    return tokens

# =========================
# 2. 篩資料
# =========================
df_kw = data.copy()
df_kw = df_kw[df_kw[LABEL_COL].isin(TARGET_LABELS)].copy()

print("資料分布：")
print(df_kw[LABEL_COL].value_counts())

# =========================
# 3. 分類詞頻
# =========================
label_tokens = defaultdict(list)

for _, row in df_kw.iterrows():
    label = row[LABEL_COL]
    tokens = tokenize(row[TEXT_COL])
    label_tokens[label].extend(tokens)

label_counter = {}
for label in TARGET_LABELS:
    label_counter[label] = Counter(label_tokens[label])

# =========================
# 4. 找通用詞
# =========================
word_in_labels = defaultdict(set)

for label in TARGET_LABELS:
    for word in label_counter[label]:
        word_in_labels[word].add(label)

common_words = set()
for w, labels in word_in_labels.items():
    if len(labels) >= 3:   # 出現在3類以上，視為通用詞
        common_words.add(w)

# =========================
# 5. 萃取每類專屬關鍵字
# =========================
label_keywords = {}

for target in TARGET_LABELS:
    scores = []

    for word, freq in label_counter[target].items():
        if word in common_words:
            continue

        other_freq = 0
        for other in TARGET_LABELS:
            if other != target:
                other_freq += label_counter[other].get(word, 0)

        score = freq / (other_freq + 1)

        if freq >= 5 or target == "Misleading":
            scores.append((word, freq, other_freq, round(score, 2)))

    scores = sorted(scores, key=lambda x: (x[3], x[1]), reverse=True)
    label_keywords[target] = scores

# =========================
# 6. 輸出結果
# =========================
for label in TARGET_LABELS:
    print(f"\n===== {label} 關鍵字 =====")
    print("(詞, 本類頻率, 其他類頻率, 分數)")
    for item in label_keywords[label][:TOP_K]:
        print(item)

資料分布：
evidence_quality
Clear         552
N/A           323
Not Clear     124
Misleading      1
Name: count, dtype: int64

===== Clear 關鍵字 =====
(詞, 本類頻率, 其他類頻率, 分數)
('退休', 23, 0, 23.0)
('建築', 21, 0, 21.0)
('台光', 20, 0, 20.0)
('傷害', 18, 0, 18.0)
('退休金', 18, 0, 18.0)
('推薦', 17, 0, 17.0)
('客服', 15, 0, 15.0)
('帳戶', 14, 0, 14.0)
('藥品', 14, 0, 14.0)
('身心', 26, 1, 13.0)
('職位', 13, 0, 13.0)
('標章', 24, 1, 12.0)
('提撥', 12, 0, 12.0)
('演練', 12, 0, 12.0)
('資恐', 12, 0, 12.0)
('引導', 11, 0, 11.0)
('失能', 11, 0, 11.0)
('青少年', 11, 0, 11.0)
('人壽', 10, 0, 10.0)
('on', 10, 0, 10.0)
('危險', 10, 0, 10.0)
('信用卡', 10, 0, 10.0)
('維修', 10, 0, 10.0)
('獎助', 10, 0, 10.0)
('心理', 10, 0, 10.0)
('洗錢', 10, 0, 10.0)
('千度', 10, 0, 10.0)
('每月', 19, 1, 9.5)
('簽署', 18, 1, 9.0)
('傷病', 9, 0, 9.0)

===== Not Clear 關鍵字 =====
(詞, 本類頻率, 其他類頻率, 分數)
('追溯', 6, 0, 6.0)
('豐田', 5, 1, 2.5)
('食品', 6, 4, 1.2)
('減塑', 5, 4, 1.0)
('年淨', 5, 5, 0.83)
('獎勵', 5, 7, 0.62)

===== Misleading 關鍵字 =====
(詞, 本類頻率, 其他類頻率, 分數)
('並兼顧', 1, 1, 0.5)
('強化供', 1,